# Phase 2: YOLOv8n baseline

1. Choose **Runtime > Change runtime type > T4 GPU**.
2. Put `constellation_resplit.zip` in a Google Drive folder named `constellation`.
3. Choose **Runtime > Run all** and allow Drive access.

If the session disconnects, run all again: training continues from the last epoch.
When it finishes, download `constellation/phase2_results.zip` from Drive.


In [ ]:
!pip -q install ultralytics==8.4.210 pyyaml
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import hashlib, pathlib, zipfile, yaml
DRIVE = pathlib.Path('/content/drive/MyDrive/constellation')
DATA = pathlib.Path('/content/data')
ZIP = DRIVE / 'constellation_resplit.zip'
assert ZIP.exists(), f'Upload the dataset zip to {ZIP}'
EXPECTED_SHA256 = '870fa2ffe1d2042a1173d40c4c75beef799b48922644a53642b3e3b18bab6424'
actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual == EXPECTED_SHA256, f'Wrong or damaged zip: {actual}'
if not (DATA / 'data.yaml').exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA)
cfg = yaml.safe_load((DATA / 'data.yaml').read_text())
cfg['path'] = str(DATA)
(DATA / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
EXPECTED_COUNTS = {'train': 3294, 'valid': 97, 'test': 92}
counts = {s: len(list((DATA / s / 'images').iterdir())) for s in EXPECTED_COUNTS}
assert counts == EXPECTED_COUNTS, counts
print('Dataset OK:', counts)


In [ ]:
%%writefile train_yolo.py
"""Fine-tune a YOLO detector on the constellation dataset.

Every experiment (baseline, fog, fog + copy-paste) uses this script, so the
only thing that differs between them is --data and --name. Early stopping is
off by default so that each run gets the same number of epochs.

Example (Colab):
  python train_yolo.py --data /content/data/data.yaml --name baseline \
      --project /content/drive/MyDrive/constellation/runs
Continue an interrupted run by repeating the command with --resume.
"""
import argparse
import hashlib
import json
import sys
import time
from pathlib import Path

import yaml


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, default=Path("data/raw/constellations/resplit/data.yaml"))
    ap.add_argument("--model", default="yolov8n.pt")
    ap.add_argument("--name", default="baseline")
    ap.add_argument("--project", type=Path, default=Path("runs/constellation"))
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--patience", type=int, default=0, help="0 turns early stopping off")
    ap.add_argument("--device", default="0", help="GPU index, or cpu")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    import torch
    import ultralytics
    from ultralytics import YOLO

    project = args.project.resolve()
    run_dir = project / args.name
    start = time.time()
    if args.resume:
        last = run_dir / "weights" / "last.pt"
        if not last.exists():
            print(f"ERROR: nothing to resume at {last}", file=sys.stderr)
            return 1
        YOLO(str(last)).train(resume=True, device=args.device, workers=args.workers)
    else:
        if run_dir.exists():
            print(f"ERROR: {run_dir} already exists. Use --resume or a new --name.", file=sys.stderr)
            return 1
        YOLO(args.model).train(
            data=str(args.data.resolve()), epochs=args.epochs, imgsz=args.imgsz,
            batch=args.batch, seed=args.seed, deterministic=True, patience=args.patience,
            device=args.device, workers=args.workers, project=str(project), name=args.name)

    best = run_dir / "weights" / "best.pt"
    if not best.exists():
        print(f"ERROR: training finished but {best} is missing", file=sys.stderr)
        return 1
    # Read the settings the run really used, which on --resume are the original
    # run's rather than this command line's.
    used = yaml.safe_load((run_dir / "args.yaml").read_text())
    info = {
        "name": args.name, "model": used["model"], "data": used["data"],
        "data_yaml_sha256": sha256(Path(used["data"])), "epochs": used["epochs"],
        "imgsz": used["imgsz"], "batch": used["batch"], "seed": used["seed"],
        "patience": used["patience"],
        "resumed": args.resume, "seconds_this_session": round(time.time() - start, 1),
        "best_sha256": sha256(best), "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (run_dir / "train_info.json").write_text(json.dumps(info, indent=2))
    print(json.dumps(info, indent=2))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile eval_yolo.py
"""Evaluate trained weights on one split and save the metrics as JSON.

--only-regex restricts the evaluation to images whose file name matches, for
example '^\\d{4}-\\d{2}-\\d{2}-' for the daily sky screenshots only.

Example:
  python eval_yolo.py --weights best.pt --data data.yaml \
      --out results/constellation/phase2/baseline_test_all.json
"""
import argparse
import hashlib
import json
import re
import sys
import tempfile
from collections import Counter
from pathlib import Path

import yaml

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--weights", type=Path, required=True)
    ap.add_argument("--data", type=Path, required=True)
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--split", default="test", choices=("train", "val", "test"))
    ap.add_argument("--only-regex", help="keep only images whose file name matches")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default="0", help="GPU index, or cpu")
    args = ap.parse_args()

    import torch
    import ultralytics
    from ultralytics import YOLO

    cfg = yaml.safe_load(args.data.read_text())
    base = Path(cfg.get("path") or args.data.parent)
    img_dir = base / cfg[args.split]
    images = sorted(p.resolve() for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS)
    if args.only_regex:
        images = [p for p in images if re.search(args.only_regex, p.name)]
    if not images:
        print(f"ERROR: no images selected from {img_dir}", file=sys.stderr)
        return 1

    names = cfg["names"]
    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    instances = Counter()
    for img in images:
        label = img.parent.parent / "labels" / f"{img.stem}.txt"
        for line in label.read_text().splitlines():
            if line.strip():
                instances[names[int(line.split()[0])]] += 1

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "images.txt").write_text("\n".join(map(str, images)) + "\n")
        run_cfg = {**cfg, "path": str(base.resolve()), args.split: str(tmp / "images.txt")}
        (tmp / "data.yaml").write_text(yaml.safe_dump(run_cfg, sort_keys=False))
        m = YOLO(str(args.weights)).val(
            data=str(tmp / "data.yaml"), split=args.split, imgsz=args.imgsz, batch=args.batch,
            device=args.device, plots=False, verbose=False, project=str(tmp / "runs"), name="val")

    box = m.box
    per_class = {c: {"instances": instances[c], "ap50": None, "ap50_95": None} for c in names}
    for i, cls in enumerate(box.ap_class_index):
        per_class[names[int(cls)]].update(ap50=float(box.ap50[i]), ap50_95=float(box.ap[i]))

    result = {
        "weights": str(args.weights),
        "weights_sha256": hashlib.sha256(args.weights.read_bytes()).hexdigest(),
        "data": str(args.data), "split": args.split, "only_regex": args.only_regex,
        "n_images": len(images), "n_instances": sum(instances.values()),
        "mAP50": float(box.map50), "mAP50_95": float(box.map),
        "precision": float(box.mp), "recall": float(box.mr), "per_class": per_class,
        "imgsz": args.imgsz, "ultralytics": ultralytics.__version__, "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(result, indent=2))
    print(json.dumps({k: result[k] for k in ("n_images", "n_instances", "mAP50", "mAP50_95",
                                             "precision", "recall")}, indent=2))
    print(f"Saved {args.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
RUNS = DRIVE / 'runs'
RUN = RUNS / 'baseline'
if (RUN / 'train_info.json').exists():
    print('Training already finished, skipping.')
else:
    resume = '--resume' if (RUN / 'weights' / 'last.pt').exists() else ''
    !python train_yolo.py --data {DATA}/data.yaml --name baseline --project {RUNS} --epochs 100 --imgsz 640 --batch 16 --seed 0 --device 0 --workers 2 {resume}
assert (RUN / 'train_info.json').exists(), 'Training did not finish; read the log above.'


In [ ]:
OUT = DRIVE / 'results' / 'phase2'
BEST = RUN / 'weights' / 'best.pt'
!python eval_yolo.py --weights {BEST} --data {DATA}/data.yaml --split test --out {OUT}/baseline_test_all.json --device 0
DAILY = r'^\d{4}-\d{2}-\d{2}-'
!python eval_yolo.py --weights {BEST} --data {DATA}/data.yaml --split test --only-regex '{DAILY}' --out {OUT}/baseline_test_daily.json --device 0
assert (OUT / 'baseline_test_all.json').exists() and (OUT / 'baseline_test_daily.json').exists()


In [ ]:
with zipfile.ZipFile(DRIVE / 'phase2_results.zip', 'w') as z:
    for p in sorted(OUT.glob('*.json')):
        z.write(p, p.name)
    for name in ('results.csv', 'args.yaml', 'train_info.json', 'weights/best.pt'):
        z.write(RUN / name, f'baseline/{name}')
print('Done. Download', DRIVE / 'phase2_results.zip')
